# 🎧 Spotify 2024 · Multi-Platform EDA

**Exploring how YouTube metrics correlate with core audio features**

This notebook performs an Exploratory Data Analysis on the *Most Streamed Spotify Songs 2024* dataset. It covers:
1. Data loading & cleaning
2. Key Performance Indicators (KPIs)
3. Feature correlation heatmap (Seaborn)
4. Audio feature landscape — KDE, box-strip & violin plots (Matplotlib + Seaborn)
5. Distribution explorer & Top-N rankings (Plotly)

---

### Dataset
Download from [Kaggle – Most Streamed Spotify Songs 2024](https://www.kaggle.com/datasets/nelgiriyewithana/most-streamed-spotify-songs-2024) and place the CSV (or ZIP) as `data/spotify_2024.csv` (or `.zip`).

## 0 · Environment Setup

In [ ]:
# Uncomment the line below to install required packages if needed
# !pip install pandas numpy matplotlib seaborn plotly

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import zipfile
import io
from pathlib import Path

# Render Plotly charts inline in the notebook
import plotly.io as pio
pio.renderers.default = "notebook"

# Increase default figure DPI for sharper matplotlib output
%matplotlib inline
plt.rcParams["figure.dpi"] = 120

print("✅ All imports loaded successfully.")

---
## 1 · Constants & Configuration
*(Originally in `data_loader.py`)*

In [ ]:
# ─── Column Constants ─────────────────────────────────────────────────────────

AUDIO_FEATURES = [
    "Danceability", "Energy", "Valence", "Acousticness",
    "Instrumentalness", "Liveness", "Speechiness",
]

# Alternate column names (some Kaggle CSVs use % suffix)
AUDIO_FEATURES_PCT = [f"{f} %" for f in AUDIO_FEATURES]
AUDIO_FEATURES_UNDERSCORE = [f"{f.lower()}_%" for f in AUDIO_FEATURES]

PLATFORM_METRICS = {
    "spotify": [
        "Spotify Streams", "Spotify Playlist Count",
        "Spotify Playlist Reach", "Spotify Popularity",
    ],
    "youtube": [
        "YouTube Views", "YouTube Likes", "YouTube Playlist Reach",
    ],
    "tiktok": [
        "TikTok Posts", "TikTok Likes", "TikTok Views",
    ],
}

ALL_PLATFORM_COLS = [c for cols in PLATFORM_METRICS.values() for c in cols]
NUMERIC_COLS = ALL_PLATFORM_COLS + AUDIO_FEATURES
ID_COLS = ["Track", "Album Name", "Artist", "Release Date", "ISRC"]

# Plotly theme
PLOTLY_LAYOUT = dict(
    template="plotly_dark",
    paper_bgcolor="rgba(0,0,0,0)",
    plot_bgcolor="rgba(0,0,0,0)",
    font=dict(family="Inter, sans-serif", color="#e0e0e0"),
    margin=dict(l=40, r=40, t=50, b=40),
)

SPOTIFY_COLORS = [
    "#1ed760", "#1db954", "#17a74a", "#14943f",
    "#b3b3b3", "#ff6b6b", "#ffd93d", "#6bcbff",
    "#c084fc", "#fb923c", "#f472b6",
]

print("✅ Constants defined.")

---
## 2 · Data Loading & Cleaning
*(Originally in `data_loader.py`)*

In [ ]:
# ─── Helper Functions ─────────────────────────────────────────────────────────

def _normalise_audio_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Map alternate audio-feature column names to canonical names."""
    for canon, pct, uscore in zip(
        AUDIO_FEATURES, AUDIO_FEATURES_PCT, AUDIO_FEATURES_UNDERSCORE
    ):
        if canon not in df.columns:
            for alt in (pct, uscore):
                if alt in df.columns:
                    df = df.rename(columns={alt: canon})
                    break
    return df


def _coerce_numeric(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Vectorised coercion of object columns to numeric."""
    present = [c for c in cols if c in df.columns]
    for c in present:
        if df[c].dtype == object:
            df[c] = df[c].astype(str).str.replace(",", "", regex=False)
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def _read_csv_from_zip(path: Path) -> pd.DataFrame:
    """Open a ZIP, find the first CSV inside, and read it."""
    with zipfile.ZipFile(path, "r") as zf:
        csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
        if not csv_names:
            raise ValueError(f"No CSV file found inside {path}")
        with zf.open(csv_names[0]) as f:
            raw = f.read()
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(io.BytesIO(raw), encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode CSV inside {path} with any known encoding.")


def load_data(path: Path) -> pd.DataFrame:
    """Read CSV (or ZIP containing a CSV) with encoding fallback."""
    path = Path(path)
    if path.suffix.lower() == ".zip":
        return _read_csv_from_zip(path)
    for enc in ("utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {path} with any known encoding.")


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Vectorised cleaning pipeline:
    1. Strip whitespace from string columns
    2. Normalise audio-feature column names
    3. Coerce numeric columns
    4. Parse release dates
    5. Drop rows that are entirely null
    6. Fill remaining NaN numerics with column median
    """
    df = df.copy()

    # 1. Strip whitespace
    str_cols = df.select_dtypes(include="object").columns
    df[str_cols] = df[str_cols].apply(lambda s: s.str.strip())

    # 2. Normalise audio feature column names
    df = _normalise_audio_cols(df)

    # 3. Coerce numeric
    df = _coerce_numeric(df, NUMERIC_COLS)

    # 4. Parse dates
    if "Release Date" in df.columns:
        df["Release Date"] = pd.to_datetime(
            df["Release Date"], errors="coerce", infer_datetime_format=True
        )
        df["Release Year"] = df["Release Date"].dt.year.astype("Int64")

    # 5. Drop all-null rows
    df = df.dropna(how="all")

    # 6. Fill missing numerics with median
    num_present = [c for c in NUMERIC_COLS if c in df.columns]
    df[num_present] = df[num_present].fillna(df[num_present].median())

    return df


def get_audio_features(df: pd.DataFrame) -> list:
    """Return the audio-feature columns that actually exist in *df*."""
    return [c for c in AUDIO_FEATURES if c in df.columns]


def get_platform_metrics(df: pd.DataFrame) -> dict:
    """Return {platform: [cols…]} for columns present in *df*."""
    return {
        plat: [c for c in cols if c in df.columns]
        for plat, cols in PLATFORM_METRICS.items()
    }

print("✅ Data loading & cleaning functions defined.")

In [ ]:
# ─── Load the Dataset ─────────────────────────────────────────────────────────

DATA_PATH = Path("data/spotify_2024.csv")
DATA_ZIP_PATH = Path("data/spotify_2024.zip")

if DATA_PATH.exists():
    raw = load_data(DATA_PATH)
elif DATA_ZIP_PATH.exists():
    raw = load_data(DATA_ZIP_PATH)
else:
    raise FileNotFoundError(
        "Dataset not found!\n"
        "Please download it from:\n"
        "  https://www.kaggle.com/datasets/nelgiriyewithana/most-streamed-spotify-songs-2024\n"
        "and place the CSV/ZIP file at 'data/spotify_2024.csv' (or .zip)."
    )

df = clean_data(raw)
print(f"✅ Dataset loaded and cleaned: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# Quick overview of the dataset
df.info()

In [ ]:
df.describe()

---
## 3 · Key Performance Indicators (KPIs)

In [ ]:
def _fmt_big(n: float) -> str:
    """Format large numbers with T/B/M/K suffix."""
    if n >= 1e12:
        return f"{n/1e12:.1f}T"
    if n >= 1e9:
        return f"{n/1e9:.1f}B"
    if n >= 1e6:
        return f"{n/1e6:.1f}M"
    if n >= 1e3:
        return f"{n/1e3:.1f}K"
    return f"{n:.0f}"


# ─── Compute & Display KPIs ───────────────────────────────────────────────────
kpis = {}

kpis["Total Tracks"] = f"{len(df):,}"

if "Spotify Streams" in df.columns:
    kpis["Total Streams"] = _fmt_big(df["Spotify Streams"].sum())
else:
    kpis["Total Streams"] = "N/A"

if "Spotify Popularity" in df.columns:
    kpis["Avg Popularity"] = f"{df['Spotify Popularity'].mean():.1f}"
else:
    kpis["Avg Popularity"] = "N/A"

if "Artist" in df.columns:
    kpis["Top Artist"] = df["Artist"].value_counts().idxmax()
else:
    kpis["Top Artist"] = "N/A"

# Display as a styled table
from IPython.display import display, HTML

kpi_html = "<table style='font-size:16px; border-collapse:collapse;'><tr>"
for label, value in kpis.items():
    kpi_html += (
        f"<td style='padding:12px 24px; text-align:center; "
        f"border:1px solid #333; background:#1a1a2e; border-radius:8px;'>"
        f"<div style='color:#b3b3b3; font-size:0.75rem; text-transform:uppercase; "
        f"letter-spacing:0.08em;'>{label}</div>"
        f"<div style='color:#1ed760; font-size:1.4rem; font-weight:700;'>{value}</div>"
        f"</td>"
    )
kpi_html += "</tr></table>"
display(HTML(kpi_html))

---
## 4 · 🔥 Feature Correlation Heatmap

Pearson correlations across streaming metrics & audio features — powered by **Seaborn**

In [ ]:
def apply_mpl_dark_theme():
    """Set a dark Spotify-flavoured theme for matplotlib/seaborn figures."""
    plt.rcParams.update({
        "figure.facecolor": "#0e1117",
        "axes.facecolor": "#0e1117",
        "axes.edgecolor": "#333333",
        "axes.labelcolor": "#e0e0e0",
        "text.color": "#e0e0e0",
        "xtick.color": "#b3b3b3",
        "ytick.color": "#b3b3b3",
        "grid.color": "#0f0f0f",
        "grid.linestyle": "--",
        "font.family": "sans-serif",
        "font.size": 11,
        "legend.facecolor": "#161b22",
        "legend.edgecolor": "#333333",
    })
    sns.set_style("darkgrid", {
        "axes.facecolor": "#0e1117",
        "figure.facecolor": "#0e1117",
        "grid.color": "#0f0f0f",
    })

apply_mpl_dark_theme()
print("✅ Dark theme applied.")

In [ ]:
# ─── Correlation Heatmap ──────────────────────────────────────────────────────

numeric_cols = [c for c in df.select_dtypes(include="number").columns
                if c != "Release Year"]

# Use up to 12 columns for a readable heatmap
selected_cols = numeric_cols[:12]

if len(selected_cols) >= 2:
    corr = df[selected_cols].corr()

    fig, ax = plt.subplots(figsize=(max(8, len(selected_cols) * 0.75),
                                    max(6, len(selected_cols) * 0.6)))

    # Custom diverging palette: teal → black → Spotify green
    cmap = sns.diverging_palette(190, 140, s=80, l=55, as_cmap=True)

    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    sns.heatmap(
        corr,
        mask=mask,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        center=0,
        vmin=-1, vmax=1,
        linewidths=0.5,
        linecolor="#1a1a2e",
        cbar_kws={"shrink": 0.8, "label": "Pearson r"},
        ax=ax,
        annot_kws={"size": 9},
    )
    ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold",
                 color="#1ed760", pad=14)
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)
    fig.tight_layout()
    plt.show()
else:
    print("⚠️ Not enough numeric columns to compute correlations.")

---
## 5 · 🎶 Audio Feature Landscape

Density curves, boxplots & violin plots for audio features — powered by **Matplotlib + Seaborn**

In [ ]:
audio_cols = get_audio_features(df)

if not audio_cols:
    print(
        "⚠️ Your dataset doesn't include audio feature columns "
        "(Danceability, Energy, Valence, …).\n"
        "Upload a version with audio features to see this section."
    )
else:
    print(f"Audio features found: {audio_cols}")

### 5.1 · KDE Density Curves

In [ ]:
if audio_cols:
    apply_mpl_dark_theme()
    palette = ["#1ed760", "#6bcbff", "#c084fc", "#ff6b6b",
               "#ffd93d", "#fb923c", "#f472b6"]

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, col in enumerate(audio_cols):
        sns.kdeplot(
            data=df, x=col, ax=ax,
            fill=True, alpha=0.25, linewidth=1.8,
            color=palette[i % len(palette)],
            label=col,
        )
    ax.set_xlabel("Feature Value", fontsize=12)
    ax.set_ylabel("Density", fontsize=12)
    ax.set_title("Audio Feature Density Curves", fontsize=14,
                 fontweight="bold", color="#1ed760", pad=12)
    ax.legend(framealpha=0.6, fontsize=9)
    fig.tight_layout()
    plt.show()
else:
    print("⚠️ Skipped — no audio feature columns found.")

### 5.2 · Box + Strip Plots

In [ ]:
if audio_cols:
    apply_mpl_dark_theme()
    melted = df[audio_cols].melt(var_name="Feature", value_name="Value")

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(
        data=melted, x="Feature", y="Value",
        palette=palette[:len(audio_cols)],
        ax=ax, linewidth=1.2, fliersize=2,
        boxprops=dict(alpha=0.7),
    )
    sns.stripplot(
        data=melted, x="Feature", y="Value",
        color="#ffffff", alpha=0.08, size=2, ax=ax, jitter=True,
    )
    ax.set_title("Audio Features — Box + Strip", fontsize=14,
                 fontweight="bold", color="#1ed760", pad=12)
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    plt.show()
else:
    print("⚠️ Skipped — no audio feature columns found.")

### 5.3 · Violin Plots

In [ ]:
if audio_cols:
    apply_mpl_dark_theme()
    melted = df[audio_cols].melt(var_name="Feature", value_name="Value")

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.violinplot(
        data=melted, x="Feature", y="Value",
        palette=palette[:len(audio_cols)],
        ax=ax, inner="quartile", linewidth=1,
    )
    ax.set_title("Audio Features — Violin Plots", fontsize=14,
                 fontweight="bold", color="#1ed760", pad=12)
    ax.tick_params(axis="x", rotation=30)
    fig.tight_layout()
    plt.show()
else:
    print("⚠️ Skipped — no audio feature columns found.")

---
## 6 · 📊 Distribution Explorer

### 6.1 · Histogram of a Key Metric

In [ ]:
# Choose a metric to explore — change this variable to explore others
DIST_METRIC = "Spotify Streams"  # Try: "Spotify Popularity", "YouTube Views", etc.

if DIST_METRIC in df.columns:
    series = df[DIST_METRIC].dropna()

    fig = px.histogram(
        series, nbins=50,
        color_discrete_sequence=["#1ed760"],
        labels={"value": DIST_METRIC},
    )
    fig.update_layout(
        **PLOTLY_LAYOUT,
        title=f"Distribution of {DIST_METRIC}",
        height=400,
        showlegend=False,
    )
    fig.show()
else:
    print(f"⚠️ Column '{DIST_METRIC}' not found in the dataset.")

### 6.2 · Violin Plot of a Key Metric

In [ ]:
if DIST_METRIC in df.columns:
    series = df[DIST_METRIC].dropna()

    fig = px.violin(
        y=series,
        color_discrete_sequence=["#1ed760"],
        box=True, points="outliers",
        labels={"y": DIST_METRIC},
    )
    fig.update_layout(
        **PLOTLY_LAYOUT,
        title=f"Violin Plot of {DIST_METRIC}",
        height=400,
        showlegend=False,
    )
    fig.show()
else:
    print(f"⚠️ Column '{DIST_METRIC}' not found in the dataset.")

---
## 7 · 🏆 Top-N Rankings

In [ ]:
# ─── Configuration ────────────────────────────────────────────────────────────
RANK_METRIC = "Spotify Streams"   # Change to rank by a different metric
TOP_N = 15                         # Number of top tracks to display

# Determine label column
label_col = "Track" if "Track" in df.columns else (
    "Track Name" if "Track Name" in df.columns else None
)

if label_col and RANK_METRIC in df.columns:
    top = df.nlargest(TOP_N, RANK_METRIC)[[label_col, "Artist", RANK_METRIC]].copy()
    top["label"] = top[label_col].str[:30] + " · " + top["Artist"].str[:20]
    top = top.sort_values(RANK_METRIC, ascending=True)

    fig = px.bar(
        top, x=RANK_METRIC, y="label",
        orientation="h",
        color=RANK_METRIC,
        color_continuous_scale=["#16213e", "#1ed760"],
        labels={RANK_METRIC: RANK_METRIC, "label": ""},
    )
    fig.update_layout(
        **PLOTLY_LAYOUT,
        title=f"Top {TOP_N} Tracks by {RANK_METRIC}",
        height=max(350, TOP_N * 28),
        coloraxis_showscale=False,
        yaxis=dict(tickfont=dict(size=11)),
    )
    fig.show()
else:
    print("⚠️ Could not find the required columns for ranking.")

---
## 8 · 🔎 Data Sample & Summary

In [ ]:
# Show a random sample of 10 tracks
df.sample(10)

In [ ]:
# Missing values summary
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing) > 0:
    print("Columns with missing values:")
    display(missing.to_frame("Missing Count"))
else:
    print("✅ No missing values in the cleaned dataset!")

---

### 📌 Credits

- **Dataset**: [Kaggle – Most Streamed Spotify Songs 2024](https://www.kaggle.com/datasets/nelgiriyewithana/most-streamed-spotify-songs-2024)
- **Libraries**: Pandas, NumPy, Matplotlib, Seaborn, Plotly
- **Original App**: Built with Streamlit (see `app.py`)